### The Winter Silence: Investigating the 2025/26 Transatlantic Mail Gap

At the end of 2025, something changed in the transatlantic mail stream. Between early November 2025 and mid-January 2026, the steady flow of postcards from Germany and Austria to the United States seemingly ground to a halt. While official postal data often remains opaque, the Postcrossing community — where German and US users form the backbone of global exchange — noticed the shift immediately.

Reports of "lost" mail began to pile up, with **travel times ballooning to over 60 days** without reaching the destination. For many, it felt as though months of holiday greetings had vanished into a "black hole." This notebook analyzes travel time distributions and volume shifts to uncover the extent of this anomaly.


In [33]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [12]:
df = pd.read_csv(
    "../data/processed/cleaned_data.csv", 
    parse_dates=['sent_date', 'received_date', 'extraction_date'],
    keep_default_na=False,  # Deactivate default NA handling --> NAMIBIA id (NA) as string
    na_values=[""]
)
print("Random sample of whole dataset:")
df.sample(10)


Random sample of whole dataset:


,country_id_dest,sent_date,received_date,distance_km,travel_time_days,extraction_date,country_id_origin,days_per_1000km,travel_route
1077166,DE,2018-09-03,2018-09-07,747,4,2026-01-21,CZ,5.35,CZ-DE
96965,CA,2013-02-15,2013-02-27,5778,12,2026-01-20,EE,2.08,EE-CA
1304308,DE,2011-11-04,2011-11-09,1179,5,2026-01-22,FI,4.24,FI-DE
1388263,NL,2015-01-23,2015-02-01,1431,9,2026-01-22,FI,6.29,FI-NL
1217451,UA,2017-05-16,2017-05-28,1662,12,2026-01-26,DE,7.22,DE-UA
923705,AU,2025-08-31,2025-09-25,15033,24,2026-01-26,BY,1.60,BY-AU
1210120,HR,2010-11-10,2010-11-17,908,7,2026-01-26,DE,7.71,DE-HR
1400761,TW,2024-07-26,2024-08-16,7969,21,2026-01-22,FI,2.64,FI-TW
1580362,PL,2012-11-25,2012-11-28,658,3,2026-01-22,NL,4.56,NL-PL
1888583,US,2020-08-22,2020-08-26,1539,4,2026-01-26,US,2.60,US-US


#### Where exactly is the US black hole?

While the "US black hole" became a heated topic primarily within the German-American community—driven by the sheer volume of active users on this route—a crucial question remains: Is this truly an isolated German-American issue, or are we looking at a much broader systemic failure?

To determine whether the silence is localized or part of a wider trend, we expanded our scope. By examining mail traffic from a selection of the world's most active regions, we can see if other global corridors are disappearing into the same void.


- USA (Domestic/Inbound)
- Canada (North America)
- Germany (Europe)
- China (Asia)
- Russia (Central Asia)
- Australia (Oceania)
- Brazil (South America)




In [ ]:
#Filtering
country_selection_continents = ["US", "DE", "CA", "CN", "BR", "AU", "RU"]
df_us = df[df["country_id_dest"] == "US"]
df_us = df_us[df_us["country_id_origin"].isin(country_selection_continents)]
df_us = df_us[df_us["received_date"] <= df_us["extraction_date"].min()] # only postcards received before earliest extraction date 
df_us = df_us[df_us["sent_date"] >= pd.Timestamp("2015-01-01")] # only postcards sent since 2015
print(f"This dataset contains {len(df_us)} unique entries sent to the US.")
print(f"The here shown postcards were received latest on {df_us["received_date"].max().date()}.")



This dataset contains 122451 unique entries sent to the US.
The here shown postcards were received latest on 2026-01-26.


In [41]:
# Travel time

color_map = {
    "US": "#EF553B",  
    "DE": "#3943C9",  
    "CA": "#00CC96",  
    "CN": "#AB63FA",  
    "BR": "#FFA15A",  
    "AU": "#19D3F3",  
    "RU": "#726E6E"    
}

monthly_data = df_us.groupby(['country_id_origin', pd.Grouper(key="sent_date", freq="MS")])['travel_time_days'].median().reset_index()
monthly_data['rolling_mean'] = monthly_data.groupby('country_id_origin')['travel_time_days'].transform(lambda x: x.rolling(window=6, center=True).mean())

fig = px.line(
    monthly_data,
    x="sent_date",
    y="rolling_mean",
    color="country_id_origin",
    title="How long do postcards travel to the US?", #rolling median
    labels={"sent_date": "Sent Date",
            "rolling_mean": "Travel Time (Median Days)",
            "country_id_origin": "Origin Country"},
    color_discrete_map=color_map,
    template="plotly_white",
    render_mode="svg"
)

fig.update_traces(mode="lines", marker=dict(size=4))
fig.update_layout(hovermode="x unified") 

fig.show();

##### From Seasonal Rhythms to Systemic Fractures

Historically, the transatlantic mail stream followed a predictable pulse. For a decade, the data showed only mild seasonal swells around the holidays, followed by a steady, gradual increase in global travel times. Even the massive disruptions of 2020 and 2021—the "COVID years"—felt like an anomaly that would eventually pass. However, as the world moved on, the old seasonal rhythms did not return; instead, they were eclipsed by deeper, more structural shifts in how mail moves across borders.

The first cracks in the system appeared between December 2023 and January 2024, when travel times spiked unexpectedly. This was the first echo of "Delivering for America," a sweeping USPS restructuring plan that left major International Service Centers (ISCs) struggling with significant backlogs. But the true turning point came in September 2024. With the implementation of the STOP Act and mandatory Electronic Advance Data (EAD) requirements, the flow of international mail hit a wall of new regulations. While most global routes eventually adapted and recovered, two corridors remained stubbornly clogged: Germany-to-US and Brazil-to-US.

The situation took a turn for the worse in May 2025. After a deceptive moment of stability during the summer, delays began to spiral again in September 2025. This time, the culprit was a "perfect storm" of stricter customs enforcement: the abolition of de-minimis rules and an aggressive crackdown on drug smuggling. These measures have pushed delivery times to record highs, leaving us with a lingering mystery: Why are Germany and Brazil bearing the brunt of these delays? To find out if this is a localized failure or a continental trend, we will now narrow our focus to the vast data landscape of the Europe-to-US route.

##### The European Fault Line: A Regional Glitch or a German Anomaly?
To understand if the "black hole" is swallowing all of Europe or merely targeting specific corridors, we must look beyond the German border. By comparing the mail volume received by US users from the most active European nations, we can pinpoint exactly where the flow of communication begins to fail.

Is the entire continent struggling under the weight of new US regulations, or is the breakdown suspiciously localized? We expanded our investigation to include the primary pulse points of European mail:

- Central Europe: Germany, Austria, Czech Republic, Poland
- Western Europe: The Netherlands, France, U.K.
- Northern Europe: Finland
- Eastern Europe & Eurasia: Belarus, Russian Federation

In [55]:
#Filtering
country_selection_europe = ["DE", "AT", "CZ", "PL", "NL", "FR", "NO", "GB", "FI", "BY", "RU"]
df_us = df[df["country_id_dest"] == "US"]
df_us = df_us[df_us["country_id_origin"].isin(country_selection_europe)]
df_us = df_us[df_us["received_date"] <= df_us["extraction_date"].min()] # only postcards received before earliest extraction date 
df_us = df_us[df_us["sent_date"] >= pd.Timestamp("2023-12-01")] # only postcards sent since 2023-12
print(f"This dataset contains {len(df_us)} unique entries sent to the US.")
print(f"The here shown postcards were received latest on {df_us["received_date"].max().date()}.")

country_selection_central = ["DE", "AT", "CZ", "PL"]
country_selection_western = ["NL", "FR", "GB"]
country_selection_northern = ["FI"]
country_selection_eurasia = ["BY", "RU"]

This dataset contains 28308 unique entries sent to the US.
The here shown postcards were received latest on 2026-01-20.


##### The Great Divergence: Mapping the 2025 Anomalies between Europe and the US

When we hold the transit data of 2025 against the mirror of the previous year, most of Europe follows a familiar, albeit slightly strained, rhythm. For the majority of european nations, the monthly median travel times fluctuated within a predictable 40% margin of 2024 levels. By the closing months of 2025, most routes had settled into a steady baseline, with variances hovering between a modest -20% and +20%. In the vast landscape of global logistics, these countries represent the "quiet" norm.

However, as we look closer, three distinct red flags emerge from the data: Belarus, Austria, and Germany. These are no longer mere fluctuations; they are systemic ruptures.

Belarus: After a relatively stable start to the year, the route hit a wall in September 2024, with travel times suddenly skyrocketing by 170%. A corridor that once functioned has effectively been throttled.

Austria: The decline was more gradual but no less severe. Starting with a minor 10–20% delay (the seemingly higher speed results from massive disruptions one year earlier.), the gap widened to +50% in May and hit a staggering +100% by August. Despite a brief, deceptive recovery, the year ended with delays surging back to 90–110% — more than doubling the expected transit time.

Germany: The epicenter of the crisis. From the very first days of 2025, German mail was already trailing by 35%. By July, the situation escalated into a full-blown logistical cardiac arrest, with delays peaking at +150%. While the numbers seemingly "improved" to a +65–85% range late in early autumn, the year 2025 ended with a massive +170% delay in November.

A Warning to the Reader: The slight downward trend visible in the final two months of our data is not necessarily a sign of recovery — it is a statistical mirage. These figures only account for the postcards that have already arrived. The "ghosts" — the thousands of cards still trapped in the machinery of the transatlantic gap — are not yet reflected in these medians. To see the true scale of the "Winter Silence," we must stop looking at how long the mail took and start looking at how much of it simply never arrived.

In [56]:
# Travel time

color_map = {
    "NL": "#EF413B",  
    "DE": "#3943C9",  
    "AT": "#00CC96",  
    "CZ": "#AB63FA",  
    "GB": "#FFA15A",  
    "PL": "#19D3F3",  
    "FI": "#F5EA50",
    "NO": "#F7BF09",
    "FR": "#FF5AD6",
    "BY": "#44454D",
    "RU": "#A7A2A2"    
}

country_dict = {"DE":"Germany", "AT": "Austria", "CZ": "Czech Republic", "PL": "Poland",
                "NL": "Netherlands", "FR": "France", "GB": "UK",
                "FI": "Finland", "NO": "Norway",
                "BY": "Belarus", "RU": "Russia"}

df_us = df_us.sort_values(by=["country_id_origin", "sent_date"])
monthly_data = df_us.groupby(['country_id_origin', pd.Grouper(key="sent_date", freq="MS")]).agg(
    travel_time_days=('travel_time_days', 'median'),
    card_count=('travel_time_days', 'count')
).reset_index()

monthly_data['prev_year_median'] = monthly_data.groupby('country_id_origin')['travel_time_days'].shift(12)
monthly_data['diff_to_prev_year'] = monthly_data["travel_time_days"] - monthly_data["prev_year_median"]
monthly_data['yoy_growth_rel'] = ((monthly_data['travel_time_days'] / monthly_data['prev_year_median']) - 1) * 100
monthly_data['country_name'] = monthly_data['country_id_origin'].map(country_dict)




In [58]:
selections = {
    "Central Europe": ["DE", "AT", "CZ", "PL"],
    "Western Europe": ["NL", "FR", "GB"],
    "Northern Europe": ["FI", "NO"],
    "Eurasia": ["BY", "RU"]
}

y_min, y_max = -80, 180

fig = make_subplots(
    rows=2, cols=2, 
    subplot_titles=list(selections.keys()),
    shared_xaxes=True, 
    shared_yaxes=True,
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

for i, (region, countries) in enumerate(selections.items()):
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    for country in countries:
        country_data = monthly_data[monthly_data["country_id_origin"] == country].dropna(subset=["yoy_growth_rel"])
        #country_name = monthly_data[monthly_data["country_id_origin"] == country]["country_name"]

        if not country_data.empty:
            fig.add_trace(
                go.Scatter(
                    x=country_data["sent_date"],
                    y=country_data["yoy_growth_rel"],
                    customdata=country_data["card_count"],
                    name=country,
                    mode="lines",
                    line=dict(color=color_map.get(country, "gray"), width=2.5),
                    hovertemplate=(
                        f"<b>{country}</b><br>" +
                        "Date: %{x|%b %Y}<br>" +
                        "Delay: %{y:.1f}%<br>" +
                        "Cards: %{customdata}<extra></extra>"
                    ),
                    showlegend=False
                ),
                row=row, col=col
            )
            
            # Country label for each line
            last_point = country_data.iloc[-1]
            fig.add_trace(
                go.Scatter(
                    x=[last_point["sent_date"]],
                    y=[last_point["yoy_growth_rel"]],
                    mode="markers+text",
                    text=[country_dict[country]],
                    textposition="middle right",
                    textfont=dict(color=color_map.get(country, "gray"), size=11, family="Arial Black"),
                    marker=dict(color=color_map.get(country, "gray"), size=5),
                    hoverinfo="skip",
                    showlegend=False
                ),
                row=row, col=col
            )
    
    # Reference lines
    fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4, row=row, col=col,
                    annotation_text="Normal" if col==1 else "", annotation_position="top right")
    fig.add_hline(y=50, line_dash="dot", line_color="red", opacity=0.4, row=row, col=col,
                    annotation_text="50% Delay" if col==1 else "", annotation_position="top right")

fig.update_layout(
    title_text="<b>How slow do postcards travel compared to one year ago?</b><br><sup>Europe to USA</sup>",
    template="plotly_white",
    hovermode="x unified",
    height=850,
    margin=dict(r=50)
)

fig.update_xaxes(range=["2025-01-01", "2026-04-01"])
fig.update_yaxes(range=[y_min, y_max], ticksuffix="%", zeroline=False)

fig.show()

##### The Vanishing Volume: Beyond the Metrics of Time
Our investigation has shown that the "black hole" is neither a blanket European crisis nor a simple regional glitch. Instead, it appears as a selective void, swallowing mail from specific origins while leaving others untouched. But travel times only tell half the story—they only account for the survivors, the cards that actually made it through. To see the true scale of the silence, we must look at the volume of mail that successfully reached US shores.

However, measuring this volume is like tracking shadows in the dark. We face several logistical uncertainties that make a simple count difficult:

The Invisible Backlog: We cannot know how many thousands of postcards are currently "in limbo"—trapped in warehouses or sorting centers—until they either arrive or expire.

Systemic Shifts: Sudden drops in volume can be triggered by more than just delays. Regulatory changes can throttle mail streams, or the underlying algorithms that assign destination addresses may shift, rerouting the flow of global exchange.

The Feedback Loop: Perhaps most critically, the "black hole" creates its own vacuum. In the Postcrossing ecosystem, a card lost in the "traveling nirvana" occupies a user’s slot for months. As more cards vanish, users are left with fewer opportunities to draw new addresses, leading to a natural—and forced—decline in new mail being sent.

What follows is an analysis of these dwindling numbers, keeping in mind that a drop in volume is often the first symptom of a system beginning to fail.